# Title

Intro

Goal

Approach

### Exec Summary

### Imports and Data Loading

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [2]:
shortlist_combined_df = pd.read_parquet("../data/processed/shortlist_combined_df.parquet")

shortlist_combined_df.columns.tolist()

['id',
 'title',
 'score',
 'upvote_ratio',
 'num_comments',
 'created_utc',
 'subreddit',
 'subscribers',
 'permalink',
 'url',
 'domain',
 'num_awards',
 'num_crossposts',
 'crosspost_subreddits',
 'post_type',
 'is_nsfw',
 'is_bot',
 'is_megathread',
 'body',
 'filename',
 'score_scaled',
 'comments_scaled',
 'engagement',
 'high_engagement',
 'engagement_rank',
 'title_length',
 'has_question',
 'has_exclamation',
 'has_number']

#### Quick recap of our dataset

Here, the Not Safe For Work flag as a proxy for a safety signal. Titles may not provide enough signal, so body content will also be considered.

In [3]:
# Inspect distribution of potential target variable
shortlist_combined_df["is_nsfw"].value_counts(normalize=True)

is_nsfw
False    0.921136
True     0.078864
Name: proportion, dtype: float64

👉 The dataset is highly imbalanced (~8% NSFW), which makes this a recall-sensitive problem. Missing unsafe content is more costly than incorrectly flagging safe content, so class weighting will be used to prioritise recall for the minority class.

In [4]:
# Check for missing values in the "body" column
shortlist_combined_df["body"].isnull().value_counts(normalize=True)

body
True     0.760813
False    0.239187
Name: proportion, dtype: float64

👉 ~ 76% of posts in the shortlist dataframe are missing body content. This could create a shortcut for the model, e.g. assuming posts with body are more likely to be unsafe. A "has body" flag can help this be monitored.

In [5]:
# Create has_body flag
shortlist_combined_df["has_body"] = ~shortlist_combined_df["body"].isnull()

shortlist_combined_df["has_body"].value_counts(normalize=True)

has_body
False    0.760813
True     0.239187
Name: proportion, dtype: float64

## Modelling

For comparison purposes, the baseline model will be limited to a feature set of titles.

The plan is then to introduce and test:
- Context (subreddit)
- Title embeddings
- Title and body embeddings

### Baseline model

In [6]:
# Create an "unsafe" flag for content that is not NSFW
shortlist_combined_df["unsafe"] = shortlist_combined_df["is_nsfw"].astype(int)
shortlist_combined_df["unsafe"].unique()

array([0, 1])

In [7]:
# Define baseline feature set of titles
X = shortlist_combined_df["title"]

# Define target variable
y = shortlist_combined_df["unsafe"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert text data to numerical features
vectorizer = TfidfVectorizer(max_features=5000)

# Train vectorizer and transform splits
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train logistic regression model with weights to handle class imbalance
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(X_train_vec, y_train)

# Predict on test set and evaluate
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.87      0.92      1279
           1       0.31      0.65      0.42       109

    accuracy                           0.86      1388
   macro avg       0.64      0.76      0.67      1388
weighted avg       0.92      0.86      0.88      1388

Confusion Matrix:
[[1119  160]
 [  38   71]]


### Title + Context

In [8]:
# Add subreddit as a feature
X_context = shortlist_combined_df[["title", "subreddit"]]
y = shortlist_combined_df["unsafe"]

# Splits
X_context_train, X_context_test, y_train, y_test = train_test_split(
    X_context,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
context_preprocessor = ColumnTransformer(
    transformers=[
        ("title_tfidf", TfidfVectorizer(stop_words='english', max_features=5000), "title"),
        ("subreddit_ohe", OneHotEncoder(handle_unknown="ignore"), ["subreddit"])
    ]
)

# Full pipeline including weighted logistic regression
context_pipeline = Pipeline(steps=[
    ("preprocessor", context_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Fit pipeline and evaluate
context_pipeline.fit(X_context_train, y_train)
y_context_pred = context_pipeline.predict(X_context_test)

# Evaluate
print("Classification Report: Title + Context")
print(classification_report(y_test, y_context_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_context_pred))

Classification Report: Title + Context
              precision    recall  f1-score   support

           0       0.97      0.85      0.91      1279
           1       0.29      0.72      0.41       109

    accuracy                           0.84      1388
   macro avg       0.63      0.78      0.66      1388
weighted avg       0.92      0.84      0.87      1388

Confusion Matrix:
[[1090  189]
 [  31   78]]


### Embeddings Safety baseline

Feature set = titles

In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
# Load embedding model
embed = SentenceTransformer("all-MiniLM-L6-v2")

# Encode titles in batches
titles = shortlist_combined_df["title"].tolist()
X_embeddings = embed.encode(
    titles,
    batch_size=32,
    show_progress_bar=True
)

print(X_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/217 [00:00<?, ?it/s]

(6936, 384)


In [11]:
# Split data
X_train_emb, X_test_emb, y_train, y_test = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train with weights
model_emb = LogisticRegression(max_iter=1000, class_weight="balanced")
model_emb.fit(X_train_emb, y_train)

y_pred_emb = model_emb.predict(X_test_emb)

# Evaluate
print("Classification Report: Title Embeddings")
print(classification_report(y_test, y_pred_emb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_emb))

Classification Report: Title Embeddings
              precision    recall  f1-score   support

           0       0.98      0.84      0.91      1279
           1       0.30      0.78      0.43       109

    accuracy                           0.84      1388
   macro avg       0.64      0.81      0.67      1388
weighted avg       0.92      0.84      0.87      1388

Confusion Matrix:
[[1079  200]
 [  24   85]]


### Embeddings and Context

In [ ]:
# --- Define new feature set with embeddings and subreddit ---

# Create column names for embedding features
embedding_dim = X_embeddings.shape[1]
embedding_cols = [f"emb_{i}" for i in range(embedding_dim)]

# Create DataFrame for embedding features
embeddings_df = pd.DataFrame(X_embeddings, columns=embedding_cols)

# Combine embeddings with subreddit
X_embed_context = pd.concat([embeddings_df, shortlist_combined_df[["subreddit"]]], axis=1)

X_embed_context.shape

(6936, 385)

In [13]:
# Split
X_embed_context_train, X_embed_context_test, y_train, y_test = train_test_split(
    X_embed_context,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocess subreddit context using one-hot encoding
embed_context_preprocessor = ColumnTransformer(
    transformers=[
        ("subreddit_ohe", OneHotEncoder(handle_unknown="ignore"), ["subreddit"]),
        ("embedding_pass", "passthrough", embedding_cols)
    ]
)

# Create pipeline for logistic regression with embedding + context features
embedding_context_pipeline = Pipeline(steps=[
    ("preprocessor", embed_context_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train model
embedding_context_pipeline.fit(X_embed_context_train, y_train)

# Evaluate model
y_embed_context_pred = embedding_context_pipeline.predict(X_embed_context_test)

print("Embedding + Context Model")
print(classification_report(y_test, y_embed_context_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_embed_context_pred))

Embedding + Context Model
              precision    recall  f1-score   support

           0       0.98      0.83      0.90      1279
           1       0.29      0.83      0.43       109

    accuracy                           0.83      1388
   macro avg       0.64      0.83      0.67      1388
weighted avg       0.93      0.83      0.86      1388

Confusion Matrix:
[[1058  221]
 [  18   91]]


Combining embeddings with subreddit context leads to the strongest safety model, achieving the highest recall for unsafe content (0.83) while also reducing false positives.

This indicates that safety is best predicted through a combination of semantic understanding of text and contextual information. Unlike engagement, where context dominated, safety benefits significantly from improved text representation.

These results highlight that different objectives (engagement vs safety) rely on different underlying signals, and that model design should reflect the specific goal being optimised.

### With body text

The final experiment tests whether adding body text improves the strongest safety model so far. Specifically, it compares title embeddings + context against combined title/body embeddings + context, holding the rest of the modelling setup constant.

In [15]:
# Create combined text feature of title + body
shortlist_combined_df["combined_text"] = shortlist_combined_df["title"] + " " + shortlist_combined_df["body"].fillna("")

# Encode combined text in batches
text = shortlist_combined_df["combined_text"].tolist()
X_text_embeddings = embed.encode(
    text,
    batch_size=32,
    show_progress_bar=True
)

print(X_text_embeddings.shape)


Batches:   0%|          | 0/217 [00:00<?, ?it/s]

(6936, 384)


In [ ]:
# Create column names for embedding features
text_embedding_dim = X_text_embeddings.shape[1]
text_embedding_cols = [f"text_emb_{i}" for i in range(text_embedding_dim)]

# Create DataFrame for text embedding features
text_embeddings_df = pd.DataFrame(X_text_embeddings, columns=text_embedding_cols)

# Create feature set with text embeddings and subreddit
X_text_embed_context = pd.concat([text_embeddings_df, shortlist_combined_df[["subreddit"]]], axis=1)

X_text_embed_context.shape

(6936, 385)

In [ ]:
# Split data
X_text_ctxt_train, X_text_ctxt_test, y_train, y_test = train_test_split(
    X_text_embed_context,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
text_preprocessor = ColumnTransformer(
    transformers=[
        ("subreddit_ohe", OneHotEncoder(handle_unknown="ignore"), ["subreddit"]),
        ("text_embedding_pass", "passthrough", text_embedding_cols)
    ]
)

# Pipeline with weighted logistic regression
text_embed_context_pipeline = Pipeline(steps=[
    ("preprocessor", text_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train model
text_embed_context_pipeline.fit(X_text_ctxt_train, y_train)

# Evaluate model
y_text_ctxt_pred = text_embed_context_pipeline.predict(X_text_ctxt_test)

print("Text Embedding + Context Model")
print(classification_report(y_test, y_text_ctxt_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_text_ctxt_pred))

Text Embedding + Context Model
              precision    recall  f1-score   support

           0       0.98      0.84      0.91      1279
           1       0.31      0.83      0.45       109

    accuracy                           0.84      1388
   macro avg       0.65      0.83      0.68      1388
weighted avg       0.93      0.84      0.87      1388

Confusion Matrix:
[[1078  201]
 [  19   90]]


Adding body text to the strongest safety model does not lead to a meaningful improvement. Recall for unsafe content remains unchanged (0.83), while false positives increase slightly.

This suggests that the most useful safety signal is already present in the title and subreddit context, and that incorporating body text adds complexity without improving performance.

## Safety threshold

Because missing unsafe content is more costly than incorrectly flagging safe content, the decision threshold can be adjusted to prioritise recall. In practice, this also opens up a more realistic moderation workflow, where borderline cases are escalated for human review rather than treated as fully safe.

In [20]:
# Get unsafe probabilities from best model so far (title embedding and context)
y_embed_context_proba = embedding_context_pipeline.predict_proba(X_embed_context_test)[:, 1]

# Test different probability thresholds for flagging content as unsafe
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for number in thresholds:
    y_pred_threshold = (y_embed_context_proba >= number).astype(int)
    print(f"\nThreshold: {number}")
    print(classification_report(y_test, y_pred_threshold))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_threshold))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.98      0.83      0.90      1279
           1       0.29      0.83      0.43       109

    accuracy                           0.83      1388
   macro avg       0.64      0.83      0.67      1388
weighted avg       0.93      0.83      0.86      1388

Confusion Matrix:
[[1058  221]
 [  18   91]]

Threshold: 0.4
              precision    recall  f1-score   support

           0       0.99      0.77      0.86      1279
           1       0.25      0.90      0.39       109

    accuracy                           0.78      1388
   macro avg       0.62      0.83      0.63      1388
weighted avg       0.93      0.78      0.83      1388

Confusion Matrix:
[[982 297]
 [ 11  98]]

Threshold: 0.3
              precision    recall  f1-score   support

           0       0.99      0.70      0.82      1279
           1       0.20      0.92      0.33       109

    accuracy                           0.71     

Adjusting the classification threshold highlights a clear trade-off between recall and false positives. Lowering the threshold significantly improves recall for unsafe content, but at the cost of a large increase in false positives (flagging content as unsafe that's actually fine):

- At a threshold of 0.5, the model misses 18 unsafe posts and flags 221 safe posts incorrectly.
- Lowering the threshold to 0.3 reduces missed unsafe posts to 9, but increases incorrect flags to 390.

This suggests that a single binary decision is insufficient for a safety-critical system. A more appropriate approach is to introduce a three-tier workflow, where high-confidence unsafe content is automatically flagged, low-risk content is allowed, and borderline cases are escalated for human review.

To make the model usable in practice, different probability thresholds can be applied depending on how risk is managed. Exact thresholds could be tuned based on moderation cost and capacity, and risk tolerance. 

For example:

- High confidence (≥0.7): automatically flag as unsafe
- Medium confidence (0.3–0.7): send for human review and moderation
- Low confidence (<0.3): allowed

In [26]:
# Save best model for future use
import joblib
joblib.dump(embedding_context_pipeline, "../models/embedding_context_safety_model.joblib")

['../models/embedding_context_safety_model.joblib']